In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

from mdx2.io import loadobj, NexusFileIndex
from mdx2.scaling import ScalingModel, OffsetModel, AbsorptionModel, DetectorModel


# Scaling model

Visualize the mdx2 scaling model.

## Parameters

In [ ]:
# These metadata fields are injected by mdx2.report, regardless of the template. Don't change this line.
_metadata: dict = {}

# Notebook parameters
input_files: list[str] = None # list of input file paths: required

In [ ]:
# pretty-print the metadata and parameters
pprint({
    "metadata": _metadata,
    "parameters": {"input_files": input_files}
})

## Results

In [ ]:
file_index = NexusFileIndex(*input_files)

### Scale

In [ ]:
for file_name, object_name in file_index.find_objects(ScalingModel):
    try:
        scaling_model = loadobj(file_name, object_name)
    except Exception as e:
        print(f"failed to load {object_name} from {file_name}: {e}")
        continue
    b = scaling_model.to_xarray()
    b.plot(aspect=1.5, size=3)
    plt.title(f"{file_name}: {object_name}")

### Offset

In [ ]:
for file_name, object_name in file_index.find_objects(OffsetModel):
    try:
        offset_model = loadobj(file_name, object_name)
    except Exception as e:
        print(f"failed to load {object_name} from {file_name}: {e}")
        continue
    c = offset_model.to_xarray()
    c.plot(aspect=1.5, size=3)
    plt.title(f"{file_name}: {object_name}")

### Absorption

In [ ]:
for file_name, object_name in file_index.find_objects(AbsorptionModel):
    try:
        absorption_model = loadobj(file_name, object_name)
    except Exception as e:
        print(f"failed to load {object_name} from {file_name}: {e}")
        continue
    a = absorption_model.to_xarray()
    phi_min = a.phi[0].data
    phi_max = a.phi[-1].data
    phi_interval = 360/8
    npts = int(1 + (phi_max - phi_min) / phi_interval)
    col_wrap = min(8, npts)
    a_sliced = a.sel(phi=np.linspace(phi_min, phi_max, npts),method='nearest')
    g = a_sliced.plot(
        yincrease=False,
        x='ix',
        y='iy',
        col='phi',
        cmap='coolwarm',
        robust=True,
        col_wrap=col_wrap,
        size=1.5,
        )
    for ax in g.axs.flatten():
        ax.set_aspect('equal')
    plt.title(f"{file_name}: {object_name}")

### Detector

In [ ]:
unique_detector_models = []
for file_name, object_name in file_index.find_objects(DetectorModel):
    try:
        detector_model = loadobj(file_name, object_name)
    except Exception as e:
        print(f"failed to load {object_name} from {file_name}: {e}")
        continue
    # first, check if this detector model is identical to any of the previously loaded ones
    if any(np.array_equal(detector_model.u, d.u) for d in unique_detector_models):
        print(f"skipping duplicate detector model {object_name} from {file_name}")
        continue
    unique_detector_models.append(detector_model)
    d = detector_model.to_xarray()
    d.plot(x='ix', y='iy', yincrease=False,cmap='coolwarm', robust=True, size=6)
    plt.title(f"{file_name}: {object_name}")
